# Results

Tables for the compaction audit harness, built from `results/summary.json` and `results/per_run.csv` (written by `scripts/summarize.py`). Nothing here talks to Modal. Run `scripts/summarize.py` and `scripts/make_figures.py` first.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

try:
    S = json.loads((ROOT / "results" / "summary.json").read_text())
    P = S["pooled"]
except (OSError, ValueError, KeyError):
    S, P = None, None
try:
    runs = pd.read_csv(ROOT / "results" / "per_run.csv")
except OSError:
    runs = None

HAVE = bool(P and P.get("n_units"))
if not HAVE:
    print("No pooled results yet. Run scripts/summarize.py after the runs are collected.")
else:
    print(f"seeds {S['meta']['seeds']}, {P['n_tasks']} tasks, {P['n_units']} (seed, task) units complete across all arms")
    print("excluded tasks:", ", ".join(S["meta"]["excluded_tasks"]))

## Main table

Only (seed, task) units complete across all five arms are counted. The difference versus `flat` is in percentage points with a 95 percent bootstrap interval over tasks.

In [ ]:
def ci(v, sign=False):
    if not v:
        return ""
    f = "{:+.1f}" if sign else "{:.1f}"
    return "[" + f.format(v[0] * 100) + ", " + f.format(v[1] * 100) + "]"

rows = []
if HAVE:
    for a in S["meta"]["arms"]:
        d = P["arms"].get(a, {})
        if not d.get("n"):
            continue
        v = P["vs_flat"].get(a)
        rows.append({
            "arm": a, "n": d["n"], "resolved": d["resolved"], "rate": d["rate"],
            "rate 95% CI": ci(d.get("rate_ci95")),
            "diff vs flat (pp)": "reference" if a == "flat" else f"{v['mean_diff'] * 100:+.1f}",
            "diff 95% CI": "" if a == "flat" else ci(v["ci95"], sign=True),
            "wins/losses": "" if a == "flat" else f"{v['wins']}/{v['losses']}",
            "gen tokens/run": d["mean_gen_tokens"], "model s/run": d["mean_model_s"],
            "step-cap runs": d["step_limit"],
        })
main = pd.DataFrame(rows)
main

## Exposure at compaction

Counts are runs per arm. `stale test` means at least one compaction retained a test record produced under an older workspace state; `stale read` is the same for read records.

In [ ]:
rows = []
if HAVE:
    for a in S["meta"]["arms"]:
        d = P["arms"].get(a, {})
        if not d.get("n"):
            continue
        rows.append({
            "arm": a, "runs": d["n"], "compacted": d["runs_compacted"], "compactions": d["compactions"],
            "ran a test": d["tasks_any_test"], "stale test": d["tasks_stale_test_at_compact"],
            "stale read": d["tasks_stale_read_at_compact"],
        })
pd.DataFrame(rows)

## Step kinds

A step is an edit if the workspace hash changed, otherwise a test if the command looks like a test run, otherwise a read.

In [ ]:
kinds = pd.DataFrame({a: d["step_kinds"] for a, d in P["arms"].items() if d.get("n")}).T.reindex(columns=["read", "edit", "test"]).fillna(0).astype(int) if HAVE else pd.DataFrame()
if len(kinds):
    kinds["total"] = kinds.sum(axis=1)
    kinds["test share"] = kinds["test"] / kinds["total"]
kinds

## Per repository

Resolved counts per arm on the units complete across all arms. `n` is the number of units in that repository.

In [ ]:
repo = pd.DataFrame(P["per_repo"]).T if HAVE else pd.DataFrame()
if len(repo):
    repo = repo[["n"] + [a for a in S["meta"]["arms"] if a in repo.columns]].astype(int)
repo

## Per seed

The same resolved counts per seed, taken from `per_run.csv` and restricted to units complete across all arms. This is a check on the pooled table, not a separate analysis.

In [ ]:
if runs is not None and len(runs):
    common = runs[runs["in_common"] == 1]
    print(f"{len(runs)} non-excluded runs in per_run.csv, {len(common)} in the common set")
    seed_tab = common.groupby(["seed", "arm"]).agg(
        n=("resolved", "size"), resolved=("resolved", "sum"),
        step_limit=("status", lambda s: int((s == "step_limit").sum())))
    seed_tab["rate"] = seed_tab["resolved"] / seed_tab["n"]
    display(seed_tab)
else:
    print("results/per_run.csv not found")

## Figures

Shown if `scripts/make_figures.py` has written PNG files. The PDF versions are in the same folder.

In [ ]:
pngs = sorted((ROOT / "figures").glob("*.png"))
if not pngs:
    print("No PNG figures yet. Run scripts/make_figures.py.")
for p in pngs:
    print(p.name)
    display(Image(filename=str(p)))